# Step 2b: clean data + WeightedRandomSampler

Step 2b: clean data (4,800 rows) + WeightedRandomSampler (class-balanced),
single-phase training. Compare against v8 (Step 2c, two-step training)
under identical eval code.

Derived from `cnn-1d-v7-clean-data.ipynb` (Step 2a). **The only difference is
the training sampler** — model, hyperparameters, `val_loader` and the shared
`validate()` call are byte-identical, so 2a / 2b / 2c are directly comparable.


In [ ]:
# v7b: trained on clean data only (n_votes>=10, 4800 rows)
# Condition 2 (Step 2b) — WeightedRandomSampler, class-balanced
# v7 architecture and hyperparameters preserved; sampler is the only change
import torch
import os
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import timm
import huggingface_hub

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt 

from timm import create_model  # or torchvision.models
from tqdm.notebook import tqdm
from torchinfo import summary

from torch.utils.data import DataLoader, random_split

In [ ]:
import sys, os

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    CODE_DIR = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code'
else:
    CODE_DIR = os.path.abspath('../project/src')

sys.path.insert(0, CODE_DIR)
sys.path.insert(0, os.path.join(CODE_DIR, 'src'))
sys.path.insert(0, os.path.abspath('../project') if not IS_KAGGLE else '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code')

import yaml
CONFIG_PATH = os.path.join(CODE_DIR, 'config.yaml') if IS_KAGGLE else '../project/config/config.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
print(cfg)

In [ ]:
# ========== ENVIRONMENT CONFIG ==========
if IS_KAGGLE:
    DATA_ROOT = '/kaggle/input/competitions/hms-harmful-brain-activity-classification'
    META_PATH = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/train_clean.csv'
else:
    DATA_ROOT = os.path.abspath('../')
    META_PATH = os.path.abspath('../kaggle_upload/train_clean.csv')

# ========== REPRODUCIBILITY ==========
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} | Device: {DEVICE}")

# v3: soft label + KL div loss + dropout (backbone=0.15, head=0.4)

In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/')
from src.data import load_clean_data, EEGDatasetV2, get_sampler, preprocess_eeg_windows
from torch.utils.data import DataLoader

CLEAN_PATH    = META_PATH  # train_clean.csv — set in environment cell above
TRAINING_MODE = "clean_weighted"  # options: clean | clean_weighted | two_step
train_df = load_clean_data(CLEAN_PATH, split='trainval')
val_df   = load_clean_data(CLEAN_PATH, split='test')

# normalize raw vote counts → probability distributions (required by KLDivLoss)
_VOTE_COLS = ['seizure_vote','lpd_vote','gpd_vote','lrda_vote','grda_vote','other_vote']
for _df in [train_df, val_df]:
    _df['target'] = _df['label']
    _vs = _df[_VOTE_COLS].sum(axis=1)
    _df['soft_y'] = list(_df[_VOTE_COLS].div(_vs, axis=0).to_numpy(dtype=np.float32))

PARQUET_DIR = os.path.join(DATA_ROOT, 'train_eegs')


In [ ]:
# ========== NPY CACHE (I/O speedup — see kaggle_run/src/data.py:preprocess_eeg_windows) ==========
cfg['cache_dir'] = cfg.get('cache_dir', '/tmp/hms_eeg_cache')
CACHE_DIR = cfg['cache_dir']
os.makedirs(CACHE_DIR, exist_ok=True)

existing = [f for f in os.listdir(CACHE_DIR) if f.endswith('.npy')]
if len(existing) == 0:
    print(f'Building EEG window cache → {CACHE_DIR}')
    preprocess_eeg_windows(pd.concat([train_df, val_df]), PARQUET_DIR, CACHE_DIR)
    print(f'Cache built: {len(os.listdir(CACHE_DIR))} files')
else:
    print(f'Cache already exists: {len(existing)} files, skipping')

train_ds = EEGDatasetV2(train_df, parquet_dir=PARQUET_DIR, cache_dir=CACHE_DIR)
val_ds   = EEGDatasetV2(val_df,   parquet_dir=PARQUET_DIR, cache_dir=CACHE_DIR)

# Step 2b: class-balanced sampling — the ONLY difference from v7 (Step 2a).
# sampler and shuffle=True are mutually exclusive, so shuffle is dropped.
sampler = get_sampler(train_df, label_col='label')
train_loader = DataLoader(train_ds, batch_size=cfg["train"]["batch_size"],
                          sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg["train"]["batch_size"],
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")
# 预期输出: Train: 4,800 | Val: 1,139

In [ ]:
# Batch shape verification
batch = next(iter(train_loader))
print("x shape:", batch["x"].shape)
print("y shape:", batch["y"].shape)
print("y sample:", batch["y"][:4])

In [ ]:
from models.classifier import build_model

model = build_model(cfg["model"]).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters — total: {total_params:,} | trainable: {trainable:,}")

In [ ]:
# ========== SHARED EVALUATION SETUP ==========
# All notebooks must score validation with the same code path so their
# numbers are directly comparable. src/evaluation.validate() divides by
# len(loader.dataset) (not len(loader)), so an uneven last batch no longer
# biases val KL; soft_key="soft_y" makes it score against the true vote
# distribution rather than a one-hot of the hard label.
from src.evaluation import validate
from src.losses import build_loss

eval_cfg = {"loss": {"name": "kl", "label_smoothing": 0.0},
            "model": {"num_classes": 6}}
# NOTE: must be the KLLoss from build_loss() — it applies log_softmax itself.
# Passing a bare nn.KLDivLoss() here would skip that and give wrong numbers.
val_loss_fn = build_loss(eval_cfg)

In [ ]:
# v4: soft label + KL div loss + dropout + bipolar montage
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
import time

NUM_EPOCHS  = cfg["train"]["epochs"]
LR          = cfg["train"]["lr"]
GRAD_CLIP   = cfg["train"]["grad_clip"]
USE_AMP     = cfg["train"]["amp"] and DEVICE.type == "cuda"

criterion = nn.KLDivLoss(reduction='batchmean')   # training loss only
optimizer = AdamW(model.parameters(), lr=LR,
                  weight_decay=cfg["train"]["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

best_kl, best_epoch, best_f1_at_best_kl, history = float('inf'), None, None, []
patience = cfg["train"].get("early_stop_patience", 10)
wait = 0

for epoch in range(1, NUM_EPOCHS + 1):
    # --- train ---
    model.train()
    t0 = time.time()
    train_loss = 0.0
    for batch in train_loader:
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(x)
            log_probs = F.log_softmax(logits, dim=1)
            loss = criterion(log_probs, batch["soft_y"].to(DEVICE))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    scheduler.step()

    # --- validate (shared: src/evaluation.validate) ---
    val_loss, metrics, val_probs, val_y_true = validate(
        model, val_loader, val_loss_fn, eval_cfg, DEVICE, soft_key="soft_y"
    )
    avg_val_kl = val_loss
    macro_f1   = metrics["macro_f1"]

    # val CE is monitoring-only; derived from the returned probs, no extra pass
    avg_val    = F.nll_loss(torch.log(val_probs.clamp_min(1e-12)), val_y_true).item()
    avg_train  = train_loss / len(train_loader)
    elapsed    = time.time() - t0

    history.append({"epoch": epoch, "train_kl": avg_train,
                    "val_kl": avg_val_kl, "val_ce": avg_val, "macro_f1": macro_f1})

    print(f"Epoch {epoch:03d} | train_kl {avg_train:.4f} | "
          f"val_kl {avg_val_kl:.4f} | val_ce {avg_val:.4f} | macro_f1 {macro_f1:.4f} | {elapsed:.0f}s")

    # checkpoint / early-stop on val_kl (the headline metric), not macro_f1
    if avg_val_kl < best_kl:
        best_kl = avg_val_kl
        best_epoch = epoch
        best_f1_at_best_kl = macro_f1
        wait = 0
        torch.save(model.state_dict(), "best_model.pt")
        print(f"  ✓ saved best model (kl={best_kl:.4f})")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nTraining complete. Best epoch {best_epoch}: "
      f"val_kl={best_kl:.4f}, macro_f1={best_f1_at_best_kl:.4f}")

In [ ]:
import matplotlib.pyplot as plt

epochs    = [h["epoch"]    for h in history]
train_kl  = [h["train_kl"] for h in history]
val_kl    = [h["val_kl"]   for h in history]
val_ce    = [h["val_ce"]   for h in history]
macro_f1  = [h["macro_f1"] for h in history]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))

ax1.plot(epochs, train_kl, label="train KL")
ax1.plot(epochs, val_kl,   label="val KL")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("KL Divergence")
ax1.set_title("KL Divergence (train vs val)"); ax1.legend()

ax2.plot(epochs, val_ce, color="orange", label="val CE")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Cross-Entropy Loss")
ax2.set_title("Val CE Loss (monitoring)"); ax2.legend()

ax3.plot(epochs, macro_f1, color="green")
ax3.set_xlabel("Epoch"); ax3.set_ylabel("Macro F1")
ax3.set_title("Validation Macro F1")

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

print(f"\nFinal epoch summary:")
print(f"  Best macro_f1 : {max(macro_f1):.4f}  (epoch {epochs[macro_f1.index(max(macro_f1))]})")
print(f"  Best val KL   : {min(val_kl):.4f}  (epoch {epochs[val_kl.index(min(val_kl))]})")
print(f"  Final train KL: {train_kl[-1]:.4f}")
print(f"  Final val KL  : {val_kl[-1]:.4f}")

In [ ]:
# One-line summary — copy into notebooks/RESULTS.md
# Reports the single checkpoint epoch (best val_kl), so val_kl and val_f1 are
# guaranteed to come from the same model — not independently-best epochs.
_best = min(history, key=lambda h: h["val_kl"])
print(f"cnn-1d-v7b-weighted-sampler | "
      f"val_kl={_best['val_kl']:.4f} | "
      f"val_f1={_best['macro_f1']:.4f} | "
      f"epoch={_best['epoch']}")